# Lab 5: Interrupts / Human-in-the-Loop (Applied Version)
In this notebook, we'll evolve our **Smart Content Generation System** to support Human Review.

Instead of automatically finishing, the workflow pauses execution *before* a publishing node. The human review node allows the human to inspect the state, edit the drafted text, set `approved = True`, and resume the run.

We use the following concepts:
- **interrupt_before**: Halting execution prior to the final approval gate.
- **get_state / update_state**: Accessing paused state information and applying manual feedback updates.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Verify API keys
print("OpenAI API Key set:", "OPENAI_API_KEY" in os.environ)
print("LangSmith tracing set:", os.environ.get("LANGCHAIN_TRACING_V2"))

OpenAI API Key set: True
LangSmith tracing set: true


### 1. Define State and Nodes

In [2]:
from typing import TypedDict
from mock_llm import get_llm
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

class PublishingState(TypedDict):
    topic: str
    draft: str
    approved: bool
    notes: str
    status: str

llm = get_llm(model="gpt-4o-mini", temperature=0.7)

def generate_draft_node(state: PublishingState):
    print("--- Node: Generating Draft ---")
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Write a brief marketing tagline (under 25 words) for a company based on the topic."),
        ("human", "Topic: {topic}")
    ])
    response = (prompt | llm).invoke({"topic": state["topic"]})
    return {"draft": response.content.strip(), "status": "drafted"}

def human_review_node(state: PublishingState):
    print("--- Node: Human Review Intercept ---")
    # Pass-through node where the graph pauses for review
    return {}

def publish_article_node(state: PublishingState):
    print("--- Node: Publishing Article ---")
    if state["approved"]:
        print(f"\n>>> PUBLISHING ADTAG: {state['draft']} <<<")
        return {"status": "published"}
    else:
        print(f"\n>>> DISCARDING ADTAG (Approved: False) <<<")
        return {"status": "rejected"}

--- OpenAI API connection failed (Error code: 429 - {'error': {'message': 'You exceeded your c...). Falling back to Mock LLM ---


### 2. Build and Compile with Interrupt before Review Node

In [3]:
builder = StateGraph(PublishingState)
builder.add_node("generate_draft", generate_draft_node)
builder.add_node("human_review", human_review_node)
builder.add_node("publish", publish_article_node)

builder.add_edge(START, "generate_draft")
builder.add_edge("generate_draft", "human_review")
builder.add_edge("human_review", "publish")
builder.add_edge("publish", END)

memory = MemorySaver()

# Pause execution right before executing 'human_review'
graph = builder.compile(
    checkpointer=memory,
    interrupt_before=["human_review"]
)

### 3. Run and Trigger Interrupt

In [4]:
config = {"configurable": {"thread_id": "review-thread-1"}}

print("Starting pipeline run...")
graph.invoke({
    "topic": "Eco-friendly reusable water bottles",
    "draft": "",
    "approved": False,
    "notes": "",
    "status": "pending"
}, config)

# Check graph status
state_snapshot = graph.get_state(config)
print("\n--- Graph Paused ---")
print("Next node scheduled to execute:", state_snapshot.next)
print("Draft to review:", state_snapshot.values["draft"])

Starting pipeline run...
--- Node: Generating Draft ---

--- Graph Paused ---
Next node scheduled to execute: ('human_review',)
Draft to review: Sip sustainably. Save our oceans.


### 4. Human Review & Resume
The human updates the state (approving it and optionally modifying the draft), and then resumes execution.

In [5]:
print("Simulating human approval...")

# Update draft text and mark approved
graph.update_state(
    config,
    {
        "approved": True,
        "draft": "Sip sustainably. Save our oceans. (Modified by Human reviewer)",
        "notes": "Approved with a slightly punchier tagline."
    },
    as_node="human_review"
)

print("\nResuming execution...")
final_state = graph.invoke(None, config)
print("\nFinal Graph State:")
print("Status:", final_state["status"])
print("Tagline:", final_state["draft"])

Simulating human approval...

Resuming execution...
--- Node: Publishing Article ---

>>> PUBLISHING ADTAG: Sip sustainably. Save our oceans. (Modified by Human reviewer) <<<

Final Graph State:
Status: published
Tagline: Sip sustainably. Save our oceans. (Modified by Human reviewer)
